# 3.7 核内切块与流水：四拍循环、乒乓与 Preload

Q、K、V 按 B、N、S1 轴核间切块后，每个核内还要对 **K、V 的 S2 轴**切块处理——这个切块原理正是 FlashAttention 算法本身（第 2 章 2.5 节）：Q 块驻留片上，KV 块逐块流入。

![核内循环](images/fia_kernel_loop.svg)

## 1. 四拍循环

每个 KV 块进来，经历四拍（AIC 与 AIV 协作）：

| 拍 | 单元 | 干什么 |
|--|--|--|
| **mm1** | AIC | Q · Kᵀ 得到 S 矩阵 |
| **vector1** | AIV | softmax、mask、scale 计算（得到 P） |
| **mm2** | AIC | P · V 得到本块输出 |
| **vector2** | AIV | 与历史状态 update（Online Softmax 修正） |

这正对应第 2 章 FA 的「两次矩阵乘夹一个 Softmax」：**mm1 → softmax → mm2 → update，循环推进直到 KV 走完**。分离架构下，Cube 与 Vector 各司其职，通过 workspace/片上缓冲交换中间数据。

## 2. MM1 的搬运与计算链路

一次 mm1（Q · Kᵀ）在存储层次间走完的全通路：

```text
Q 矩阵：  GM --DataCopy--> L1A --LoadData--> L0A
K 矩阵：       GM -->     L1B --LoadData--> L0B
                                        │
                          mmad：L0A × L0B → L0C
                                        │
                          FixPipe：L0C → UB（交给 vector1）
```

每一段对应第 1 章的搬运单元分工：

| 段 | 单元 | 说明 |
|--|--|--|
| GM → L1 | MTE2 | 外部数据搬进片上 |
| L1 → L0 | MTE1 | 供 Cube 的 A/B 操作数就位 |
| L0C → UB | FIXPIPE | 矩阵乘结果交付 Vector |

细节：**左矩阵**搬运 L1→L0 用 `LoadData`；**右矩阵**（如 K）搬运 L1→L0 使用 `LoadDataWithTranspose` 接口——搬运的同时完成转置，免得单独算转置矩阵。

## 3. MM2 的搬运与计算链路

```text
vector1 的结果 P：UB --> L1A --LoadData--> L0A
V 矩阵：          GM --> L1B --LoadDataWithTranspose--> L0B
                                        │
                          mmad：L0A × L0B → L0C
                                        │
                          FixPipe：L0C → UB（交给 vector2 update）
```

注意方向变化：**vector1 的结果从 UB 回流到 L1A**，作为 mm2 的左矩阵——这是分离架构下 Vector → Cube 的数据通路（借助 910D 起打通的 UB ↔ L1 双向通路）。

## 4. Vector 侧实现

- Vector 核寄存器大小为 **256B**，一条 VF 指令可以处理 **64 个 FP32** 数据；
- **vector1 计算的基本块大小为 64 × 128 个 FP32 数据**；
- 采用 **Regbase 编程**（基于寄存器基址直接编排数据）。

对照 Tiling ⑤：SoftMax 的 shape 是 `{s1BasicBlock/nRatio, s2BasicBlock×nRatio}`——**nRatio 调整 Vector 侧块的纵横比**，把它与 Cube 侧的 `s1 块 × (s2 块×nRatio)` 大块配平，让一次 Cube 计算量正好覆盖若干个 Vector 基本块。

## 5. 流水并行：乒乓与 Preload

如果四拍严格串行，搬运和计算互相等待，大量时间浪费在空转。两种武器：

### 乒乓内存

把缓冲区分成 A/B 两份：算第 i 块时，第 i+1 块的数据已经在另一份里搬运。**搬运与计算在时间上重叠**，任何时刻都有一份数据在被计算、另一份在被填充。

### Preload

**提前启动 mm1 和 vector1**：不等当前拍算完，下一拍需要的 KV 块的 mm1/vector1 就提前发起，通过 `taskId` 与 `PRELOAD_N` 参数控制流水节奏。

```text
串行：      [搬运|mm1|vec1|mm2|vec2] [搬运|mm1|vec1|mm2|vec2] ...
乒乓+preload: [搬运|mm1|vec1|mm2|vec2]
                  [搬运|mm1|vec1|mm2|vec2]     ← 与上一块重叠
                       [搬运|mm1|vec1|mm2|vec2] ← 与上上块重叠
```

> 这也解释了 Tiling ⑥ 里 workspace 的「倍数」：stage1 / stage2 各留 2~4 倍空间，正是给乒乓与流水预留的缓冲。

## 6. 把三层串起来

至此，一次 FA 调用的完整链路：

```text
接口层：  aclnn 两段式，参数检查 + executor 构造
            │
Tiling：  模板匹配 → 基本块 → 核间切分 → API 排产 → workspace → TilingKey
            │  （产出 TilingData + blockDim，全部在 Host/CPU 完成）
            ▼
Kernel：  SPMD 多组合核启动
            ├─ 核间：按 B/N2/S1 切块 + 负载均衡（splitFactorSize 份/核）
            └─ 核内：沿 S2 切块，四拍循环（mm1 → vector1 → mm2 → vector2）
                     乒乓内存 + preload 流水
```

第 4 章将把这套代码真正跑起来（基于 CANNSim 仿真），第 5 章再从零写一个基础极简 FA。

## 本节小结

- 核内沿 **S2 轴**切块，四拍循环 = **mm1（Q·Kᵀ）→ vector1（softmax/mask/scale）→ mm2（·V）→ vector2（update）**。
- MM1/MM2 各有完整搬运链路：GM → L1（MTE2）→ L0（MTE1）→ 计算 → L0C → UB（FIXPIPE）；右矩阵用 `LoadDataWithTranspose` 顺带转置。
- Vector 一条 VF 指令 64 个 FP32（寄存器 256B），vector1 基本块 64×128；nRatio 负责配平 Cube 与 Vector。
- 乒乓内存 + preload（taskId / PRELOAD_N 控制）让搬运与计算重叠，workspace 的倍数就是流水线的缓冲代价。

## 小测验

1. 四拍循环中，Q 和 P 分别做 mm1、mm2 的左矩阵，它们的来源有什么不同？（各从哪里出发）
2. 为什么右矩阵搬运 L1→L0 用 `LoadDataWithTranspose`？
3. 乒乓内存解决什么问题？如果没有乒乓，「搬运」和「计算」在时间上是什么关系？
4. vector1 基本块是 64×128，一条 VF 指令处理 64 个 FP32，处理完一个基本块至少需要发多少条 VF 指令（仅按数据量估算）？